In [ ]:
# Verify Tensorflow installation and version
import tensorflow as tf
print(tf.version.VERSION)

# Loading in the datasets
import glob
import numpy as np

# For the 75/25 split
from sklearn.model_selection import train_test_split

# Confusion Matrix plotting
import matplotlib.pyplot as plt
import seaborn as sns

### Loading in the data

Datasets are stored in the project 'dataset' folder in sub-folders for each difficulty class (defined as 'labels' in the code below). 

**X = 1000 (SUBJECT TO CHANGE BASED ON HOW WE ARE GENERATING DATA)**

Each file has X rows.  When you add a new motion classification (in your second pass through this programming assignment) make sure your new datasets have X rows (not X-1, or X+1) to match this format and make sure you store them in a sub-folder with the same name as your new label in the 'labels' array in the code below.

This code builds two data structures: x_recordings[] and y_recordings[].  

x_recordings is a three-dimensional array.  The first dimension is a dataset index. The second dimension is the dataset's temporal dimension, and the third dimension indexes the 5x IMUs, each with axis: accelX, accelY, accelZ, gyroX, gyroY, gyroZ.

**Y = 93 (SUBJECT TO CHANGE BASED ON HOW WE ARE GENERATING DATA)**

y_recordings[] contains the Y labels ('easy', 'medium', 'hard') associated each of the Y datasets. 

When you add new datasets later on, the number Y should grow by however many new datasets you add, but the x_recordings[] second (4500) and third (30) dimensions should not chanage.

In [ ]:
# Load data into memory
labels = ['easy', 'medium', 'hard']
x_recordings = []
y_recordings = []
recordings_filenames = []
for i, label in enumerate(labels):
    filenames = glob.glob('dataset/' + label + '/*.csv')
    for filename in filenames:
        data = np.loadtxt(filename, delimiter=',')
        x_recordings.append(data)
        y_recordings.append(i)
        recordings_filenames.append(filename)

x_recordings = np.array(x_recordings).reshape(len(x_recordings), -1, 30)
y_recordings = np.array(y_recordings)

print(x_recordings.shape)
print(y_recordings.shape)

### View the data???

### Splitting it up into windows???

### Pre-Processing???

### 75/25 Split

In [ ]:
# Shuffle the dataset and then split with 75% for training and 25% for testing
x_train, x_test, y_train, y_test = train_test_split(
    x_frames_normed, y_frames, test_size=0.25)

print("Trainning samples:", x_train.shape)
print("Testing samples:", x_test.shape)

### Creating and training the model

In [ ]:
## Conv1D based model
model = tf.keras.models.Sequential([
  tf.keras.Input(shape=(4500, 30)),
  tf.keras.layers.Conv1D(filters=16, kernel_size=7, padding="same", use_bias=False),
  tf.keras.layers.BatchNormalization(), # is often paired with use_bias=False
  tf.keras.layers.ReLU(),
  tf.keras.layers.MaxPool1D(pool_size=4),

  tf.keras.layers.Conv1D(filters=32, kernel_size=5, padding="same", use_bias=False),
  tf.keras.layers.BatchNormalization(),
  tf.keras.layers.ReLU(),
  tf.keras.layers.MaxPool1D(pool_size=4),
  
  tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding="same", use_bias=False),
  tf.keras.layers.BatchNormalization(),
  tf.keras.layers.ReLU(),
  tf.keras.layers.MaxPool1D(pool_size=4),

  tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding="same", use_bias=False),
  tf.keras.layers.BatchNormalization(),
  tf.keras.layers.ReLU(),
  tf.keras.layers.GlobalAveragePooling1D(),

  tf.keras.layers.Dropout(0.4),
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dropout(0.3),
  tf.keras.layers.Dense(3, activation='softmax')
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy', 'precision', "F1Score"])

model.fit(x_train, y_train, epochs=30)
test_loss, test_acc = model.evaluate(x_test,  y_test, verbose=2)

print("Test loss:", test_loss)
print("Test acc:", test_acc)
model.summary()

### Confusion Matrix
Now we will run predictions using the reserved test data to see how well the training worked. Generate a heat map showing right/wrong guesses vs. truth by motion class.

In [ ]:
# Evaluate the training parameters on test data
%matplotlib inline

Y_pred = model.predict(x_test)
y_pred = np.argmax(Y_pred, axis=1)
confusion_matrix = tf.math.confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(confusion_matrix,
            annot=True,
            xticklabels=labels,
            yticklabels=labels,
            cmap=plt.cm.Blues,
            fmt='d', cbar=False)
plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

### Saving the model
Save it to both a 'standard' file format (.h5) and to a 'lightweight' format for use with the microcontroller (.tflite)

In [ ]:
# Save the model and move it to the microcontroller
model.save('model.keras')

# Convert to tflite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
          f.write(tflite_model)